In [ ]:
!pip install scikit-learn

In [ ]:
# ============================================================================
# Kable: Epistemic Reasoning & Theory of Mind — Kaggle Benchmark Task
# ============================================================================
# Task: Given a logical scenario involving recursive knowledge, determine 
#       the correct belief attribution.
# Input: A "query" string containing the scenario and multiple choice options.
# Output: (A), (B), or (C)
# Metric: Accuracy (percentage of correct logical attributions)
# Samples: 250 (Stratified across logic categories, deterministic)
# ============================================================================

import kaggle_benchmarks as kbench
from dataclasses import dataclass
from sklearn.metrics import accuracy_score
import pandas as pd
import json
import os
import re

# ============================================================================
# CONFIGURATION
# ============================================================================

DATASET_ROOT = "/kaggle/input/datasets/udaysonawane/kable-dataset/kable-dataset"
NUM_SAMPLES = 250
RANDOM_SEED = 42

# ============================================================================
# STRUCTURED OUTPUT SCHEMA
# ============================================================================

@dataclass
class KableAnswer:
    """The LLM must return the multiple-choice letter."""
    answer: str


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def load_kable_dataset(root_path):
    """Iterate through .jsonl files and aggregate scenarios."""
    all_rows = []
    if not os.path.exists(root_path):
        raise FileNotFoundError(f"Directory not found: {root_path}")

    # Ensure deterministic file order for reproducibility
    for filename in sorted(os.listdir(root_path)):
        if filename.endswith(".jsonl"):
            file_path = os.path.join(root_path, filename)
            with open(file_path, "r") as f:
                for line in f:
                    try:
                        all_rows.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue
    
    return pd.DataFrame(all_rows)


def select_stratified_samples(df, n=NUM_SAMPLES):
    """
    Selects N samples via stratified sampling across experiment categories.
    Fixes the Pandas FutureWarning by using include_groups=False.
    """
    if len(df) == 0:
        raise ValueError("Dataset is empty.")

    # Calculate base samples per category
    categories = df["experiment_setup"].unique()
    samples_per_cat = n // len(categories)
    
    # Stratified selection
    # include_groups=False fixes the FutureWarning in Pandas 2.2.0+
    selected_df = (
        df.groupby("experiment_setup", group_keys=False)
        .apply(lambda x: x.sample(min(len(x), samples_per_cat), random_state=RANDOM_SEED), 
               include_groups=False)
    )
    
    # Fill remaining slots to reach exactly N if integer division left a gap
    if len(selected_df) < n:
        diff = n - len(selected_df)
        extra = df[~df.index.isin(selected_df.index)].sample(n=diff, random_state=RANDOM_SEED)
        selected_df = pd.concat([selected_df, extra])
        
    # Sort by index to keep the sample order consistent across different model runs
    return selected_df.sort_index().reset_index(drop=True)


def normalize_answer(raw_prediction):
    """Extracts (A), (B), or (C) from various response formats."""
    # Look for (A), (B), or (C)
    match = re.search(r"\(([A-C])\)", raw_prediction.upper())
    if match:
        return f"({match.group(1)})"
    
    # Fallback: Look for the letter A, B, or C standing alone or at the end
    match_bare = re.search(r"\b([A-C])\b", raw_prediction.upper())
    if match_bare:
        return f"({match_bare.group(1)})"
        
    return "INVALID"


# ============================================================================
# MAIN BENCHMARK TASK
# ============================================================================

@kbench.task(
    name="kable_logic_reasoning",
    description=(
        "Evaluates Theory of Mind and recursive knowledge. Tests how models "
        "track beliefs through multiple layers of attribution (James knows Mary knows...)."
    ),
)
def kable_benchmark(llm) -> float:
    # 1. Load and sample
    full_df = load_kable_dataset(DATASET_ROOT)
    samples = select_stratified_samples(full_df, n=NUM_SAMPLES)

    y_true = []
    y_pred = []
    total = len(samples)

    print(f"\n[BENCHMARK STARTED] Total Samples: {total}")
    print("-" * 50)

    # 2. Execution Loop
    for i, (_, row) in enumerate(samples.iterrows()):
        ground_truth = row["answer"]
        query = row["query"]
        
        # Use specific chat contexts for each sample to avoid "memory" leakage
        with kbench.chats.new(f"sample_{i}"):
            try:
                prediction = llm.prompt(query, schema=KableAnswer)
                raw_output = prediction.answer
            except Exception:
                raw_output = llm.prompt(query)

        predicted = normalize_answer(raw_output)
        
        y_true.append(ground_truth)
        y_pred.append(predicted)

        # Log progress every 50 samples to avoid excessive scroll in Kaggle UI
        if (i + 1) % 50 == 0 or (i + 1) == total:
            current_acc = accuracy_score(y_true, y_pred)
            print(f"Progress: {i+1}/{total} | Current Accuracy: {current_acc:.2%}")

    # 3. Final Evaluation
    final_accuracy = accuracy_score(y_true, y_pred)
    
    print("\n" + "=" * 50)
    print(f"FINAL RESULT: {final_accuracy:.4f}")
    print("=" * 50)

    # 4. Mandatory Kaggle Assertions
    kbench.assertions.assert_true(
        final_accuracy >= 0.0, 
        expectation="Accuracy score must be a valid percentage."
    )
    
    invalid_rate = y_pred.count("INVALID") / total
    kbench.assertions.assert_true(
        invalid_rate < 0.2,
        expectation=f"Model format failure rate is too high ({invalid_rate:.1%})."
    )

    return float(final_accuracy)


# ============================================================================
# RUN
# ============================================================================

kable_benchmark.run(kbench.llm)